# Creating and Using Catalogs

In [ ]:
import ocean_skill as osk
import ocean_skill
from ocean_skill.build import build_kerchunk, build_catalog, discover_opendap_files
from ocean_skill.obs.modis import nickname
from pathlib import Path


## Creating Catalogs

### For Datasets

#### CalOOS

In [ ]:
# %%time

# server = "https://sensors.erddap.caloos.org/erddap/"
# # server = "https://sensors.erddap.caloos.org/erddap/tabledap/wmo_46013.html"

# import intake_erddap
# discovered = intake_erddap.ERDDAPCatalogReader(server=server, query_type="intersection",
#                                                cdm_data_type="timeseries",
#                                                search_for=["timeSeries","-timeSeriesProfile","-NWIS"]).read()
#                                                # search_for=["timeSeries","-timeSeriesProfile"], standard_names=["sea_water_temperature"]).read()
# build_catalog(discovered, "catalogs/caloos_timeseries.yaml", title="CalOOS TimeSeries Datasets", probe=True, skip_errors=True)

In [ ]:
list(discovered)

#### Mixed Layer Depth (Holte and Talley)

In [ ]:
from ocean_skill.build import new_catalog, add_source

cat = new_catalog()
add_source(
    cat,
    "argo_mld",
    "https://mixedlayer.ucsd.edu/data/Argo_mixedlayers_monthlyclim_04142022.nc",
    reader_kwargs={"engine": "scipy"},  # netCDF3 classic — h5netcdf can't read it
)

# ds = cat["argo_mld"].read()  # lazy, dask-backed — no local download

In [ ]:
from ocean_skill import build

build.build_catalog(
    {
        # Direct single-file netCDF over HTTPS — simplest case
        "Holte Talley Argo MLD monthly climatology": {
            "url": "https://mixedlayer.ucsd.edu/data/Argo_mixedlayers_monthlyclim_04142022.nc",
            # provenance / algorithm metadata (extra keys land in entry metadata)
            "doi": "10.1002/2017GL073426",
            "mld_methods": [
                "density_algorithm", "density_threshold",
                "temperature_algorithm", "temperature_threshold",
            ],
            "threshold_density": 0.03,   # kg/m3, ref 10 dbar
            "threshold_temperature": 0.2,  # degC, ref 10 dbar
        },
    },
    "catalogs/mld_climatologies.yaml",
    reader_kwargs={"engine": "scipy"},  # netCDF3 classic — h5netcdf can't read it
    title="Global mixed layer depth climatologies",
    name_map=None,  # rely on the files' own attrs, not the ROMS map
)

In [ ]:
osk.read("Holte Talley Argo MLD monthly climatology")

In [ ]:
# from ocean_skill import build

# build.build_catalog(
#     {
#         # Direct single-file netCDF over HTTPS — simplest case
#         "Holte Talley Argo MLD monthly climatology": {
#             "url": "https://mixedlayer.ucsd.edu/data/Argo_mixedlayers_monthlyclim_04142022.nc",
#             # provenance / algorithm metadata (extra keys land in entry metadata)
#             "doi": "10.1002/2017GL073426",
#             "mld_methods": [
#                 "density_algorithm", "density_threshold",
#                 "temperature_algorithm", "temperature_threshold",
#             ],
#             "threshold_density": 0.03,   # kg/m3, ref 10 dbar
#             "threshold_temperature": 0.2,  # degC, ref 10 dbar
#         },
#         # # SEANOE ships a .tar of netCDFs — same reader GLODAP uses
#         # "de Boyer Montegut MLD monthly climatology v2023": {
#         #     "url": "https://www.seanoe.org/data/00806/91774/data/103667.tar",
#         #     "reader": "ocean_skill.readers:PoochTarNetCDF",
#         #     "reader_kwargs": {"member_glob": "*.nc"},
#         #     "doi": "10.17882/91774",
#         #     "mld_methods": ["density_threshold"],
#         #     "threshold_density": 0.03,   # kg/m3, ref 10 m
#         # },
#     },
#     "catalogs/mld_climatologies.yaml",
#     title="Global mixed layer depth climatologies",
#     name_map=None,  # rely on the files' own attrs, not the ROMS map
# )

#### WHOTS time series

takes about 15 min

In [ ]:
%%time

from ocean_skill.build import discover_opendap_files, build_catalog

# Walk the NCEI THREDDS directory (the browser catalog URL pastes in as-is)
# and make one entry per file; nicknames come straight from the filenames.
files = discover_opendap_files(
    "https://www.ncei.noaa.gov/thredds-ocean/catalog/ndbc/oceansites/DATA/WHOTS/catalog.html"
)
build_catalog(
    {f.rsplit("/", 1)[-1].removeprefix("OS_").removesuffix(".nc"): f for f in sorted(files)},
    "catalogs/whots.yaml",
    title="WHOTS mooring (OceanSITES)",
    skip_errors=True,  # one flaky OPeNDAP URL shouldn't discard 400+ probed entries
)

#### WOA

In [ ]:
# WOA23 — many OPeNDAP URLs, one set of shared reader options
BASE = "https://www.ncei.noaa.gov/thredds-ocean/dodsC/woa23/DATA"
CODE = {"nitrate": "n", "phosphate": "p", "silicate": "i", "oxygen": "o"}
build_catalog(
    {f"woa23_{var}_month{m:02d}": f"{BASE}/{var}/netcdf/all/1.00/woa23_all_{c}{m:02d}_01.nc"
     for var, c in CODE.items() for m in range(1, 13)},
    "catalogs/woa.yaml", title="WOA23",
    reader_kwargs={"decode_times": False, "engine": "h5netcdf", "chunks": {}},
)

#### OOI Papa

In [ ]:
import intake_erddap
discovered = intake_erddap.ERDDAPCatalogReader(server="https://erddap.dataexplorer.oceanobservatories.org/erddap", search_for=["Papa"]).read()
build_catalog(discovered, "catalogs/ooi_papa2.yaml", title="OOI Station Papa2", probe=True)

In [ ]:
print(osk.describe("OOI Station Papa2"))

In [ ]:
print(osk.describe("ooi-gp02hypm-rim01-02-ctdmog039"))

In [ ]:
osk.read("ooi-gp02hypm-rim01-02-ctdmog039")

In [ ]:

cat = intake_erddap.ERDDAPCatalogReader(
    server="https://erddap.dataexplorer.oceanobservatories.org/erddap",
    search_for=["Papa"]
)
cat.read()
build_catalog(cat, discovered, probe=True)

In [ ]:
list(cat)

In [ ]:
cat._entries

In [ ]:
# OOI Station Papa — ERDDAP tabledap, one entry per instrument
SERVER = "https://erddap.dataexplorer.oceanobservatories.org/erddap"
ids = ["ooi-gp03flma-ris01-05-flortd000b", "ooi-gp03flmb-ris01-05-flortd000b"]
build_catalog(
    {i: {"reader": "intake_erddap.erddap:TableDAPReader",
         "reader_kwargs": {"server": SERVER, "dataset_id": i,
                           "protocol": "tabledap", "mask_failed_qartod": True}}
     for i in ids},
    "catalogs/ooi_papa2.yaml", title="OOI Station Papa2",
)

#### GLODAP

In [ ]:
# GLODAP — a remote tarball of per-variable NetCDFs
build_catalog(
    {"glodap": {
        "reader": "ocean_skill.readers:PoochTarNetCDF",
        "reader_kwargs": {
            "url": "https://www.nodc.noaa.gov/archive/arc0107/0162565/2.2/data/"
                   "0-data/mapped/GLODAPv2.2016b_MappedClimatologies.tar.gz",
            "member_glob": "*.nc",
            "var_from_filename": True,      # variable name comes from the filename
            "keep_vars": ["Depth"],
            "cache_dir": "~/.ocean-skill/cache/obs",
        }}},
    "catalogs/glodap.yaml", title="GLODAP v2.2016b",
)

#### OceanSODA

In [ ]:
LOC = "https://www.ncei.noaa.gov/data/oceans/ncei/ocads/data/0220059/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"

build_catalog(
    {"oceansoda3": f"blockcache::{LOC}"},
    "catalogs/oceansoda3.yaml",
    title="OceanSODA3",
    storage_options={"blockcache": {"cache_storage": "~/.ocean-skill/cache/obs",
                                    "check_files": False}},
)

In [ ]:
osk.read("OceanSODA")

In [ ]:
osk.read("OceanSODA:OceanSODA")

In [ ]:
osk.read("OceanSODA2:OceanSODA2")

In [ ]:
build_catalog(
    {"OceanSODA2": "https://www.ncei.noaa.gov/data/oceans/ncei/ocads/data/0220059/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"},
    "catalogs/oceansoda2.yaml",
    title="OceanSODA2",
)

In [ ]:
loc = "https://www.ncei.noaa.gov/data/oceans/ncei/ocads/data/0220059/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"

cat = new_catalog(title="OceanSODA")
add_source(cat, "OceanSODA", f"simplecache://::{loc}", storage_options={"simplecache": {"same_names": True}})
save(cat, "catalogs/oceansoda.yaml")


#### Satellite data

In [ ]:
Copernicus

In [ ]:
%%time

import warnings
from ocean_skill.build import new_catalog, add_copernicus_source, save

# nickname base -> Copernicus Marine dataset_id
# Each becomes two entries: "<name>_timeseries" (timeChunked, fast point/time-series)
# and "<name>_geo" (geoChunked, fast spatial maps). No product_id needed anymore.
DATASETS = {
    "ssh_duacs_my_daily":      "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",
    "ssh_duacs_my_monthly":    "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1M-m",
    "ssh_duacs_nrt_daily":     "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D",
    "ssh_duacs_twosat_daily":  "c3s_obs-sl_glo_phy-ssh_my_twosat-l4-duacs-0.25deg_P1D",
    "cur_multiobs_my_daily":   "cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m",
    "cur_multiobs_nrt_daily":  "cmems_obs-mob_glo_phy-cur_nrt_0.25deg_P1D-m",
    "chl_gapfree_my_daily":    "cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D",
    "chl_gapfree_nrt_daily":   "cmems_obs-oc_glo_bgc-plankton_nrt_l4-gapfree-multi-4km_P1D",
    "chl_climatology_doy":     "cmems_obs-oc_glo_bgc-plankton_my_l4-multi-climatology-4km_P1D",
    "transp_gapfree_my_daily": "cmems_obs-oc_glo_bgc-transp_my_l4-gapfree-multi-4km_P1D",
    "pp_my_monthly":           "cmems_obs-oc_glo_bgc-pp_my_l4-multi-4km_P1M",
    "optics_my_monthly":       "cmems_obs-oc_glo_bgc-optics_my_l4-multi-4km_P1M",
    "sst_ostia_nrt_daily":     "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2",
    "sst_c3s_rep_daily":       "C3S-GLO-SST-L4-REP-OBS-SST",
    "sst_esacci_rep_daily":    "ESACCI-GLO-SST-L4-REP-OBS-SST",
    "wind_my_hourly":          "cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
    "wind_nrt_hourly":         "cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H",
    "sss_my_weekly":           "cmems_obs-mob_glo_phy-sss_my_multi-oi_P1W",
    "swh_my_daily":            "cmems_obs-wave_glo_phy-swh_my_multi-l4-0.5deg_P1D-i",
    "glorys_my_daily":         "cmems_mod_glo_phy_my_0.083deg_P1D-m",
    "glorys_climatology":      "cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m",
    "glo_anfc_daily":          "cmems_mod_glo_phy_anfc_0.083deg_P1D-m",
}

cat = new_catalog(title="Copernicus Marine")
for name, dataset_id in DATASETS.items():
    # add each layout separately so one missing layout can't abort the whole build
    for service in ("arco-time-series", "arco-geo-series"):
        try:
            add_copernicus_source(cat, name, dataset_id, services=(service,))
        except Exception as e:  # e.g. a dataset that doesn't publish that chunking
            warnings.warn(f"skipped {name} [{service}]: {e}")

save(cat, "catalogs/copernicus.yaml")

In [ ]:
# old version

# DATASETS = {   # nickname: (product_id, dataset_id)
#     "ssh_duacs_my_daily":      ("SEALEVEL_GLO_PHY_L4_MY_008_047",
#                                "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D"),
#     "ssh_duacs_my_monthly":    ("SEALEVEL_GLO_PHY_L4_MY_008_047",
#                                "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1M-m"),
#     "ssh_duacs_nrt_daily":     ("SEALEVEL_GLO_PHY_L4_NRT_008_046",
#                                "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D"),
#     "ssh_duacs_twosat_daily":  ("SEALEVEL_GLO_PHY_CLIMATE_L4_MY_008_057",
#                                "c3s_obs-sl_glo_phy-ssh_my_twosat-l4-duacs-0.25deg_P1D"),
#     "cur_multiobs_my_daily":   ("MULTIOBS_GLO_PHY_MYNRT_015_003",
#                                "cmems_obs-mob_glo_phy-cur_my_0.25deg_P1D-m"),
#     "cur_multiobs_nrt_daily":  ("MULTIOBS_GLO_PHY_MYNRT_015_003",
#                                "cmems_obs-mob_glo_phy-cur_nrt_0.25deg_P1D-m"),
#     "chl_gapfree_my_daily":    ("OCEANCOLOUR_GLO_BGC_L4_MY_009_104",
#                                "cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D"),
#     "chl_gapfree_nrt_daily":   ("OCEANCOLOUR_GLO_BGC_L4_NRT_009_102",
#                                "cmems_obs-oc_glo_bgc-plankton_nrt_l4-gapfree-multi-4km_P1D"),
#     "chl_climatology_doy":     ("OCEANCOLOUR_GLO_BGC_L4_MY_009_104",
#                                "cmems_obs-oc_glo_bgc-plankton_my_l4-multi-climatology-4km_P1D"),
#     "transp_gapfree_my_daily": ("OCEANCOLOUR_GLO_BGC_L4_MY_009_104",
#                                "cmems_obs-oc_glo_bgc-transp_my_l4-gapfree-multi-4km_P1D"),
#     "pp_my_monthly":           ("OCEANCOLOUR_GLO_BGC_L4_MY_009_104",
#                                "cmems_obs-oc_glo_bgc-pp_my_l4-multi-4km_P1M"),
#     "optics_my_monthly":       ("OCEANCOLOUR_GLO_BGC_L4_MY_009_104",
#                                "cmems_obs-oc_glo_bgc-optics_my_l4-multi-4km_P1M"),
#     "sst_ostia_nrt_daily":     ("SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001",
#                                "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2"),
#     "sst_c3s_rep_daily":       ("SST_GLO_SST_L4_REP_OBSERVATIONS_010_024",
#                                "C3S-GLO-SST-L4-REP-OBS-SST"),
#     "sst_esacci_rep_daily":    ("SST_GLO_SST_L4_REP_OBSERVATIONS_010_024",
#                                "ESACCI-GLO-SST-L4-REP-OBS-SST"),
#     "wind_my_hourly":          ("WIND_GLO_PHY_L4_MY_012_006",
#                                "cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H"),
#     "wind_nrt_hourly":         ("WIND_GLO_PHY_L4_NRT_012_004",
#                                "cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H"),
#     "sss_my_weekly":           ("MULTIOBS_GLO_PHY_SSS_L4_MY_015_015",
#                                "cmems_obs-mob_glo_phy-sss_my_multi-oi_P1W"),
#     "swh_my_daily":            ("WAVE_GLO_PHY_SWH_L4_MY_014_007",
#                                "cmems_obs-wave_glo_phy-swh_my_multi-l4-0.5deg_P1D-i"),
#     "glorys_my_daily":         ("GLOBAL_MULTIYEAR_PHY_001_030",
#                                "cmems_mod_glo_phy_my_0.083deg_P1D-m"),
#     "glorys_climatology":      ("GLOBAL_MULTIYEAR_PHY_001_030",
#                                "cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m"),
#     "glo_anfc_daily":          ("GLOBAL_ANALYSISFORECAST_PHY_001_024",
#                                "cmems_mod_glo_phy_anfc_0.083deg_P1D-m"),
# }

# import json, urllib.request
# from ocean_skill.build import build_catalog

# STAC = "https://stac.marine.copernicus.eu/metadata"

# def fetch(u):
#     with urllib.request.urlopen(u, timeout=90) as fh:
#         return json.load(fh)

# def arco_url(product_id, dataset_id, chunking="timeChunked"):
#     """Current ARCO URL. Resolved rather than hardcoded because CMEMS embeds a
#     version tag (..._202411) that changes when a product is republished."""
#     col = fetch(f"{STAC}/{product_id}/product.stac.json")
#     versioned = next(l["href"].split("/")[0] for l in col["links"]
#                      if l.get("rel") == "item"
#                      and l["href"].split("/")[0].rsplit("_", 1)[0] == dataset_id)
#     item = fetch(f"{STAC}/{product_id}/{versioned}/dataset.stac.json")
#     return next(a["href"] for a in item["assets"].values()
#                 if a.get("type") == "application/vnd+zarr"
#                 and a["href"].endswith(f"{chunking}.zarr"))

# urls = {k: arco_url(*v) for k, v in DATASETS.items()}
# build_catalog(urls, "catalogs/copernicus.yaml", title="Copernicus Marine",
#               reader_kwargs={"zarr_format": 2, "chunks": {}}, 
#               # skip_errors=True
#              )

Coastwatch

In [ ]:
from ocean_skill.build import build_catalog

ERDDAP = "https://coastwatch.pfeg.noaa.gov/erddap"

IDS = [
    # SST
    "jplMURSST41", "jplMURSST41mday", "jplMURSST41anom1day", "jplMURSST41clim",
    "jplMURSST42", "nesdisGeoPolarSSTN5NRT", "ncdcOisst21Agg", "ncdcOisst21NrtAgg",
    "nceiErsstv5", "erdMH1sstd1day_R2022SQMasked", "erdMH1sstd1day_R2022NRTMasked",
    "nesdisVHNsstDaily", "erdMWsstd1day", "erdMBsstd1day",
    # chlorophyll
    "nesdisVHNSQchlaDaily", "nesdisVHNSQchlaMonthly",
    "noaacwNPPN20S3ASCIDINEOF2kmDaily", "nesdisVHNnoaaSNPPnoaa20chlaGapfilledDaily",
    "nesdisVHNnoaaSNPPnoaa20NRTchlaGapfilledDaily",
    "erdMH1chla1day_R2022SQ", "erdMH1chla1day_R2022NRT", "erdMWchla1day",
    # optics / biogeochemistry
    "nesdisVHNSQkd490Daily", "nesdisVHNSQkdparDaily",
    "noaacwNPPN20S3AkdSCIDINEOF2kmDaily", "noaacwNPPN20S3AspmSCIDINEOF2kmDaily",
    "erdMPIC1day_R2022SQ", "erdMPIC1day_R2022NRT",
    "erdMPOC1day_R2022SQ", "erdMPOC1day_R2022NRT",
    "erdMH1cflh1day_R2022SQ", "erdMH1cflh1day_R2022NRT",
    "productivity_viirs_snpp_daily", "productivity_viirs_snpp_nrt_daily",
    # physical
    "erdQCwindproducts1day", "pifscCcmpDailyV21NRT", "coastwatchSMOSv662SSS1day",
    "nsidcG02202v6nh1day", "nsidcG02202v6sh1day",
]

# EDDGridFromErddap proxies: PFEG serves their metadata but 302-redirects data reads
# to PacIOOS, and netCDF-C does not follow it. Point at the source.
ELSEWHERE = {
    "NOAA_DHW":         "https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km",
    "NWW3_Global_Best": "https://pae-paha.pacioos.hawaii.edu/erddap/griddap/ww3_global",
}

urls = {i: ELSEWHERE.get(i, f"{ERDDAP}/griddap/{i}") for i in IDS + list(ELSEWHERE)}

build_catalog(urls, "catalogs/coastwatch.yaml", title="NOAA CoastWatch",
              reader_kwargs={"chunks": "auto"}, skip_errors=True,
             )

#### MODIS AQUA

Sources
* https://www.earthdata.nasa.gov/data/catalog/ob-cloud-modisa-l3m-chl-2022.0
    * 2002-07 to present
    * available per file over opendap
* 


From 2002 on are




LET"S GET ALL YEARS IN HERE! OR ONE CAT PER YEAR?

In [ ]:
import numpy as np

years = np.arange(2002, 2027)

for year in years:

    cat_name = "catalogs/modis_aqua.yaml"
    if Path(cat_name).exists():
        print(f"Catalog {cat_name} already exists, skipping")
    else:
        files = discover_opendap_files("http://oceandata.sci.gsfc.nasa.gov/opendap/MODISA/L3SMI/2003/", pattern="*.chlor_a.9km.nc", recurse=True, max_dirs=5000)  
        build_catalog({nickname(f): f for f in files if nickname(f)}, cat_name,
                    title="MODIS Aqua", storage_options={"simplecache": {"same_names": True}})

In [ ]:
osk.find("MODIS")

### For Model Output

## Set up catalog for model output

In [ ]:
# For Ben runs
# had to convert grid file to netcdf4 for this to work
refs = build_kerchunk({"dev": "dev/joined_output/*.nc", "dev_marblsub": "dev_marblsub/joined_output/*.nc"},
                      root="/anvil/projects/x-ees250129/x-bsaenz/saved_runs/", 
                      grid="/home/x-kthyng/packages/ocean-skill/localfiles/cson_roms-marbl_v0_1_Gulf_of_Alaska_64procs_grid_netcdf4.nc", 
                      out_dir="/home/x-kthyng/packages/ocean-skill/localfiles/")
build_catalog(refs, "catalogs/ben_dev.yaml", title="Bens comparison runs")

In [ ]:
# For Ben runs
# tanagra
refs1 = build_kerchunk({"dev": "cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs_20100101-20100701_dev/joined_output/*.nc", 
                       "dev_marblsub": "cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs_20100101-20100701_dev_marblsub/joined_output/*.nc"},
                      root="/mnt/scratch/saved_runs/", 
                      grid="/mnt/scratch/cstar-forge-data/cson_roms-marbl_v0.1_Gulf_of_Alaska_64procs/input_data/cson_roms-marbl_v0_1_Gulf_of_Alaska_64procs_grid.nc", 
                      out_dir="/home/kthyng/projects/ocean-skill/localfiles/")

refs2 = build_kerchunk({"ccs": "output_rst.*.nc", 
                       "ccs_cdr": "output_cdr.??????????????.nc"},
                      root="/mnt/cooltub/cw/user/blsaenz/cstar-roms-run/cson_roms-marbl_v0.1_ccs-4km_128procs_128cores/joined_output/", 
                      grid="/mnt/cooltub/cw/user/blsaenz/cstar-roms-run/cson_roms-marbl_v0.1_ccs-4km_128procs_128cores/input/input_datasets/cson_roms-marbl_v0_1_ccs-4km_128procs_grid.nc", 
                      out_dir="/home/kthyng/projects/ocean-skill/localfiles/")


# build_catalog(refs, "catalogs/ben_dev_tanagra.yaml", title="Bens comparison runs on tanagra")

In [ ]:
build_catalog(refs1 | refs2, "catalogs/ben_dev_tanagra.yaml", title="Bens comparison runs on tanagra")

In [ ]:
# build refs for BGC and HIS files separately, and make a catalog with each as a separate source

refs = build_kerchunk({"dev": "output_bgc.*.nc", "GOM_his": "output_his.*.nc"},
                      root="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/output_dt450/", 
                      grid="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/input/input_datasets/cson_roms-marbl_v0_1_GOM-Offline_noPIO_6procs_grid.nc", 
                      out_dir="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/output_dt450")
build_catalog(refs, "catalogs/gom.yaml", title="GOM offline run")

In [ ]:
ls /Volumes/storage

In [ ]:
build_roms_catalog(run_dir, "catalogs/gom.yaml", prefix="GOM", title="GOM offline run")
# -> refs/GOM_bgc.parquet, refs/GOM_his.parquet -> entries GOM_bgc, GOM_his

In [ ]:
# build refs for BGC and HIS files separately, and make a catalog with each as a separate source

refs = build_kerchunk({"GOM_bgc": "output_bgc.*.nc", "GOM_his": "output_his.*.nc"},
                      root="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/output_dt450/", 
                      grid="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/input/input_datasets/cson_roms-marbl_v0_1_GOM-Offline_noPIO_6procs_grid.nc", 
                      out_dir="/Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/output_dt450")
build_catalog(refs, "catalogs/gom.yaml", title="GOM offline run")

In [ ]:
refs

In [ ]:
ls /Users/kthyng/cstar-forge-data/cstar-blueprint-run/cson_roms-marbl_v0.1_GOM-Offline_noPIO_6procs/output_dt450

In [ ]:
build_kerchunk?

In [ ]:
build_catalog?

## Using Catalogs

### Map locations from catalogs

In [ ]:
osk.map_datasets(catalog="OOI*")         # one catalog
osk.map_datasets(renderer="holoviews")

In [ ]:
osk.find(variable="nitrate").map()

### Look at catalogs

In [ ]:
osk.catalogs

### List sources across catalogs

In [ ]:
list(osk.catalogs)

In [ ]:
osk.catalogs.names()

### List sources in one catalog

### Search across catalogs

In [ ]:
TEMP = "sea_water_potential_temperature"
osk.find(standard_name=TEMP)

### Examine metadata in a catalog

In [ ]:
print(osk.describe("glodap"))

In [ ]:
print(osk.describe("GOM_bgc"))

In [ ]:
print(osk.describe("GOM offline run"))

### Examine metadata in a source within a catalog